[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/33.%20The%20Labor%20Shift%20Scheduling%20Problem/P33-Tier-7_executed.ipynb)

---


# 33. The Labor Shift Scheduling Problem

## Tier 7: The Human–AI Symbiotic Partnership

### Goal

This notebook presents a small, rule-based scheduling demonstration and discusses how human judgment could complement automated recommendations. The implemented scheduler assigns available employees to each day's shift requirements in a fixed order. It is a teaching example, not an optimization model or a complete human–AI collaboration system.

The example helps distinguish an automated recommendation from a genuine human–AI workflow: a collaboration system would need recorded human decisions, explanations, feedback, and measured outcomes. Those components are discussed here but are not implemented by the scheduling code.

### Inputs and outputs

The inputs are a set of employees, five workdays, three shifts per day, randomly generated employee availability, and a demand value for each day–shift pair. The scheduler returns the assignments it can make under its availability and one-shift-per-employee-per-day rules. It also reports coverage, the number of assignments, and unfilled demand.

### Model assumptions and scope

- Availability is generated independently for each employee and day–shift pair, with probability 0.8. Since no random seed is set, the generated scenario and its results may differ between runs.
- Each employee can be assigned to at most one shift per day. The example does not model shift start and end times, rest periods, weekly hour limits, skills, preferences, labor regulations, or labor costs.
- Employees are considered in their input order. The scheduler selects the first available employees until demand is met or no eligible employee remains. This is a greedy rule, not an optimization algorithm; it does not search for a globally best schedule.
- The reported AI accuracy, satisfaction, collaboration efficiency, and trust values are illustrative placeholders. They are not computed from assignments, human feedback, or observed outcomes.

### Human–AI collaboration: concepts and example

A human–AI partnership can combine automated data processing with contextual knowledge that is difficult to encode, such as an employee's circumstances or an upcoming special event. Explainable recommendations can help a manager understand why a schedule was proposed and decide whether to accept or revise it. To evaluate whether collaboration actually helps, a system would need to record recommendations, human decisions, and meaningful performance measures over time.

The FashionForward retail-chain scenario describes 240 stores and proposes the following expected outcomes: a 28% improvement in employee satisfaction, a 15% reduction in labor costs, a 22% decrease in turnover, a 95% manager adoption rate, and schedule completion that is 31% faster. These are scenario claims, not results produced or validated by this notebook. The scenario assigns foot-traffic analysis, skills-based matching, cost optimization, and conflict detection to AI; it assigns employee preferences, event planning, training coordination, and customer relationships to managers.

### Scheduling rule in mathematical form

Let \(E\) be the set of employees, \(D\) the set of days, and \(S\) the set of shifts. For employee \(e\), day \(d\), and shift \(s\), let \(a_{eds}\) equal 1 if the employee is available and 0 otherwise. Let \(r_{ds}\) be the required number of employees for that day and shift. The binary decision variable \(x_{eds}\) equals 1 when employee \(e\) is assigned to that shift, and 0 otherwise.

The schedule must respect availability, the one-shift-per-day rule, and demand as an upper limit on assignments:

$$x_{eds} \leq a_{eds} \quad \forall e \in E, d \in D, s \in S,$$
$$\sum_{s \in S} x_{eds} \leq 1 \quad \forall e \in E, d \in D,$$
$$\sum_{e \in E} x_{eds} \leq r_{ds} \quad \forall d \in D, s \in S.$$

The implementation uses a fixed-order greedy rule to choose assignments satisfying these restrictions; it does not solve an optimization problem. For a completed schedule, coverage is the number of assignments divided by total demand, provided total demand is positive. Because at most one assignment per employee per day is allowed, total assignments cannot exceed \(|E||D|\). Unfilled demand is reported rather than treated as a violation of a requirement to cover every slot.

### What to examine

Check how availability and the one-shift-per-day rule limit assignments, and compare assignments with demand. The plots distinguish quantities calculated from the generated schedule from the illustrative collaboration metrics. Neither the code nor the plots establish that human–AI collaboration improves satisfaction, trust, cost, or scheduling speed.

In [ ]:
import random

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("default")
sns.set_palette("husl")

In [ ]:
class Employee:
    """Represent an employee and their availability for each day–shift pair."""

    def __init__(self, id, name, department=None):
        self.id = id
        self.name = name
        self.department = department or "General"
        self.availability = self._generate_availability()
        self.assigned_shifts = 0

    def _generate_availability(self):
        days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
        shifts = ["Morning", "Afternoon", "Night"]
        return {
            (day, shift): random.random() < 0.8
            for day in days
            for shift in shifts
        }

    def is_available(self, day, shift):
        return self.availability.get((day, shift), False)


class Shift:
    """Represent a named shift."""

    def __init__(self, id, name):
        self.id = id
        self.name = name


class HumanAICollaborationSystem:
    """Run the example's fixed-order, availability-based scheduling rule."""

    def __init__(self, employees, shifts, days, demand):
        self.employees = employees
        self.shifts = shifts
        self.days = days
        self.demand = demand
        self.weekly_schedule = {}

        # These illustrative values are not estimated from schedule or feedback data.
        self.performance_metrics = {
            "ai_accuracy": 0.75,
            "human_satisfaction": 4.2,
            "collaboration_efficiency": 0.85,
            "trust_level": 0.75,
        }

        print("Scheduling demonstration initialized")

    def run_weekly_scheduling(self):
        print("\n" + "=" * 60)
        print("RULE-BASED WEEKLY SCHEDULING DEMONSTRATION")
        print("=" * 60)

        weekly_schedule = {}

        for day in self.days:
            assigned_employee_ids = set()

            for shift in self.shifts:
                required = self.demand[day][shift.name]
                available = [
                    employee
                    for employee in self.employees
                    if employee.id not in assigned_employee_ids
                    and employee.is_available(day, shift.name)
                ]
                selected = available[:required]

                for employee in selected:
                    weekly_schedule[(employee.id, day, shift.name)] = employee
                    assigned_employee_ids.add(employee.id)
                    employee.assigned_shifts += 1

        total_assignments = len(weekly_schedule)
        total_required = sum(
            self.demand[day][shift.name]
            for day in self.days
            for shift in self.shifts
        )
        unfilled_demand = total_required - total_assignments
        coverage_rate = (
            total_assignments / total_required
            if total_required > 0
            else 0.0
        )

        print(f"Coverage rate: {coverage_rate:.1%}")
        print(f"Assignments made: {total_assignments}")
        print(f"Unfilled demand: {unfilled_demand}")
        print("Illustrative AI accuracy placeholder: "
              f"{self.performance_metrics['ai_accuracy']:.1%}")
        print("Illustrative satisfaction placeholder: "
              f"{self.performance_metrics['human_satisfaction']:.1f}/5.0")
        print("Illustrative collaboration-efficiency placeholder: "
              f"{self.performance_metrics['collaboration_efficiency']:.1%}")
        print("Illustrative trust placeholder: "
              f"{self.performance_metrics['trust_level']:.1%}")

        self.weekly_schedule = weekly_schedule
        self.total_required = total_required
        self.total_assignments = total_assignments
        self.unfilled_demand = unfilled_demand

        assert total_assignments <= total_required
        assert unfilled_demand == total_required - total_assignments
        assert all(
            employee.is_available(day, shift_name)
            for (employee_id, day, shift_name), employee
            in weekly_schedule.items()
        )
        assert len({(employee_id, day) for employee_id, day, _ in weekly_schedule}) == total_assignments

        return weekly_schedule


def create_scenario():
    """Create the employees, shifts, workdays, and demand for the example."""
    employees = [
        Employee(i + 1, f"Employee_{i + 1:03d}")
        for i in range(20)
    ]

    shifts = [
        Shift(1, "Morning"),
        Shift(2, "Afternoon"),
        Shift(3, "Night"),
    ]

    days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]

    demand = {
        "Monday": {"Morning": 8, "Afternoon": 10, "Night": 4},
        "Tuesday": {"Morning": 7, "Afternoon": 9, "Night": 3},
        "Wednesday": {"Morning": 8, "Afternoon": 10, "Night": 4},
        "Thursday": {"Morning": 7, "Afternoon": 9, "Night": 3},
        "Friday": {"Morning": 9, "Afternoon": 12, "Night": 5},
    }

    return employees, shifts, days, demand


employees, shifts, days, demand = create_scenario()

print("Scheduling scenario created:")
print(f"Employees: {len(employees)}")
print(f"Shifts per day: {len(shifts)}")
print(f"Days: {len(days)}")
print(
    "Total demand slots: "
    f"{sum(demand[day][shift.name] for day in days for shift in shifts)}"
)

collaboration_system = HumanAICollaborationSystem(
    employees, shifts, days, demand
)
schedule = collaboration_system.run_weekly_scheduling()


In [ ]:
def visualize_collaboration_metrics(collaboration_system):
    """Plot schedule-derived results separately from illustrative placeholders."""
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    days = collaboration_system.days
    shifts = collaboration_system.shifts
    demand_by_shift = {
        shift.name: [
            collaboration_system.demand[day][shift.name]
            for day in days
        ]
        for shift in shifts
    }
    assigned_by_shift = {
        shift.name: [
            sum(
                1
                for employee_id, assigned_day, shift_name
                in collaboration_system.weekly_schedule
                if assigned_day == day and shift_name == shift.name
            )
            for day in days
        ]
        for shift in shifts
    }

    x_positions = list(range(len(days)))
    width = 0.25
    for index, shift in enumerate(shifts):
        offset = (index - (len(shifts) - 1) / 2) * width
        axes[0, 0].bar(
            [x + offset for x in x_positions],
            demand_by_shift[shift.name],
            width=width,
            label=f"{shift.name} demand",
            alpha=0.45,
        )
        axes[0, 0].bar(
            [x + offset for x in x_positions],
            assigned_by_shift[shift.name],
            width=width,
            label=f"{shift.name} assigned",
            edgecolor="black",
            linewidth=0.5,
        )

    axes[0, 0].set_title("Demand and assignments by day and shift")
    axes[0, 0].set_xlabel("Day")
    axes[0, 0].set_ylabel("Employees")
    axes[0, 0].set_xticks(x_positions)
    axes[0, 0].set_xticklabels(days, rotation=25, ha="right")
    axes[0, 0].legend(fontsize=8, ncol=2)
    axes[0, 0].grid(axis="y", alpha=0.3)

    coverage = (
        collaboration_system.total_assignments
        / collaboration_system.total_required
        if collaboration_system.total_required
        else 0.0
    )
    axes[0, 1].bar(
        ["Assigned", "Unfilled"],
        [
            collaboration_system.total_assignments,
            collaboration_system.unfilled_demand,
        ],
        color=["steelblue", "darkorange"],
    )
    axes[0, 1].set_title(f"Demand coverage: {coverage:.1%}")
    axes[0, 1].set_ylabel("Demand slots")
    axes[0, 1].grid(axis="y", alpha=0.3)

    metrics = collaboration_system.performance_metrics
    axes[1, 0].bar(
        ["AI accuracy", "Collaboration efficiency", "Trust"],
        [
            metrics["ai_accuracy"],
            metrics["collaboration_efficiency"],
            metrics["trust_level"],
        ],
        color=["steelblue", "seagreen", "indianred"],
    )
    axes[1, 0].set_title("Illustrative placeholders (not measured)")
    axes[1, 0].set_ylabel("Placeholder value (0–1)")
    axes[1, 0].set_ylim(0, 1)
    axes[1, 0].grid(axis="y", alpha=0.3)

    axes[1, 1].bar(
        ["Human satisfaction"],
        [metrics["human_satisfaction"]],
        color="mediumpurple",
    )
    axes[1, 1].set_title("Illustrative placeholder (not measured)")
    axes[1, 1].set_ylabel("Placeholder score (out of 5)")
    axes[1, 1].set_ylim(0, 5)
    axes[1, 1].grid(axis="y", alpha=0.3)

    fig.suptitle("Scheduling results and illustrative collaboration metrics")
    fig.tight_layout()
    plt.show()


visualize_collaboration_metrics(collaboration_system)


## Summary and interpretation

The code produces a feasible set of assignments for the stated availability, demand, and one-shift-per-day rules. Coverage and unfilled demand are calculated from those assignments. Because the scheduler uses a fixed employee order and makes local choices shift by shift, its output depends on that order and on the randomly generated availability; it is not evidence of a globally optimal schedule.

The plotted demand and assignment counts describe the generated schedule. The AI accuracy, collaboration efficiency, trust, and satisfaction values are placeholders, not measurements. No human decisions or feedback are collected, so this example cannot demonstrate learning, explainability, improved satisfaction, or the benefits claimed in the FashionForward scenario.

### Strengths and limitations of the demonstration

- **Strengths:** The assignment rule is easy to inspect, availability is respected, duplicate daily assignments are prevented, and unfilled demand is visible.
- **Limitations:** The fixed-order selection rule does not optimize a stated objective. The example omits employee preferences, skills, shift timing, labor rules, and costs. Its collaboration metrics are illustrative rather than empirically grounded.

A fuller human–AI scheduling workflow would require clearly defined human review and feedback, measurable performance criteria, and explanations tied to actual recommendations. Such components are outside the implemented scope of this notebook.